![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 23 — Lab 1: Learn Without Labels (SimCLR)

You have a pile of images but **no labels**. Can you still learn useful features?

**SimCLR** says yes — train a model to recognize that two augmented views of the **same** image should have similar representations, while different images should be pushed apart.

<center><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/simclr_contrastive_learning.png?raw=1" width="500px"></center>

You will:
- Define a **data-augmentation** pipeline
- Build a custom **PairDataset** that returns two views of each image
- Construct a SimCLR encoder (ResNet-18 backbone + projection head)
- Implement **InfoNCE loss** from scratch using cosine similarity
- Compare: training from scratch vs contrastive pretraining — who learns better features?

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

plt.rcParams['figure.dpi'] = 120
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Data

We use **CIFAR-10** — 60,000 tiny 32×32 colour images in 10 classes. For contrastive pretraining we **throw away the labels** and treat the images as unlabeled.

In [ ]:
# --- GIVEN ---
cifar_raw  = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transforms.ToTensor())
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())

class_names = cifar_raw.classes

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = cifar_raw[i * 3000]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(class_names[label], fontsize=8)
    ax.axis('off')
fig.suptitle('CIFAR-10 samples — for contrastive learning we ignore the labels', fontsize=12)
plt.tight_layout()
plt.show()

---
## Augmentations

The augmentations are the **most important** design decision. They define what the model treats as "the same image" — if an augmented view looks very different from the original but came from the same photo, the model must learn deep features (shape, object identity) rather than shallow ones (colour, position).

<center><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/simclr_augmentations.png?raw=1" width="500px"></center>

We keep one view as a simple **identity** (just normalize) and the other gets the full augmentation pipeline. This way the model learns: "the augmented version should map close to the clean original."

In [ ]:
# ============================================================
# TODO: Fill in the augmentation parameters
# ============================================================

# View 1: identity — just convert to tensor and normalize
identity_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

# View 2: strong augmentation
aug_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=None,   # TODO
            contrast=None,     # TODO
            saturation=None,   # TODO
            hue=None,          # TODO
        )
    ], p=0.8),
    transforms.RandomGrayscale(p=None),  # TODO
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

---
## Pair Dataset

For contrastive learning, each sample must return **two views** of the same image — the clean original and an augmented version. We build a custom `Dataset` for this — the same `__getitem__` / `__len__` pattern you already know.

In [ ]:
# ============================================================
# TODO: Complete the PairDataset
# ============================================================

class PairDataset(Dataset):
    """Returns (identity view, augmented view) of the same image."""
    def __init__(self, dataset, id_transform, aug_transform):
        self.dataset = dataset
        self.id_transform = id_transform
        self.aug_transform = aug_transform

    def __len__(self):
        return None  # TODO

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]          # ignore the label
        img = transforms.ToPILImage()(img)  # transforms expect a PIL image
        view1 = None  # TODO: apply self.id_transform to img  (clean view)
        view2 = None  # TODO: apply self.aug_transform to img (augmented view)
        return view1, view2


# --- use a 20K subset for speed ---
pair_dataset = PairDataset(Subset(cifar_raw, range(20_000)), identity_transform, aug_transform)
pair_loader  = DataLoader(pair_dataset, batch_size=256, shuffle=True, num_workers=0, drop_last=True)

print(f'Pair dataset: {len(pair_dataset):,} images → each returns (clean, augmented)')

In [ ]:
# --- GIVEN: visualize some pairs ---

def denorm(t, mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]):
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    return (t * s + m).clamp(0, 1)

fig, axes = plt.subplots(4, 2, figsize=(4, 8))
for row in range(4):
    v1, v2 = pair_dataset[row * 500]
    axes[row, 0].imshow(denorm(v1).permute(1, 2, 0))
    axes[row, 0].axis('off')
    axes[row, 1].imshow(denorm(v2).permute(1, 2, 0))
    axes[row, 1].axis('off')
    if row == 0:
        axes[row, 0].set_title('Original')
        axes[row, 1].set_title('Augmented')
fig.suptitle('Positive pair — model learns: augmented ≈ original', fontsize=11)
plt.tight_layout()
plt.show()

---
## SimCLR Encoder

The encoder has two parts:

1. **Backbone** — a ResNet-18 without its final classification layer. It outputs a representation vector **h**.
2. **Projection head** — a small MLP that maps **h** → **z**. The contrastive loss is computed on **z**, but we keep **h** for downstream tasks.

<center><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/simclr_network_setup.svg?raw=1" width="350px"></center>

> **Hint:** check what `resnet18` outputs after removing the final FC layer.

In [ ]:
# ============================================================
# TODO: Fill in the layer dimensions
# ============================================================

class SimCLREncoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # remove final FC

        self.projector = nn.Sequential(
            nn.Linear(None, None), nn.ReLU(),  # TODO
            nn.Linear(None, None),              # TODO
        )

    def forward(self, x):
        h = self.backbone(x).flatten(1)   # representation we keep
        z = self.projector(h)              # projection used only for the loss
        return h, z

---
## InfoNCE Loss

There is no `nn.InfoNCE` in PyTorch — you build it yourself. Here is the equation:

$$\mathcal{L}_i \;=\; -\log \frac{\exp\!\bigl(\cos(z_i,\, z_i^{+}) \,/\, \tau\bigr)}{\displaystyle\sum_{j=1}^{N} \exp\!\bigl(\cos(z_i,\, z_j) \,/\, \tau\bigr)}$$

Read it out loud:
- **Numerator:** how similar is $z_i$ to its positive partner $z_i^+$?
- **Denominator:** sum of similarities to *all* $z_j$ in the batch
- We want the positive similarity to **dominate** → push positives together, push negatives apart

### The key insight

This is just a **softmax** over cosine similarities, where the correct class is the positive pair. And that is exactly what `F.cross_entropy` computes!

Let's visualize the similarity matrix for a mini-batch to see this clearly.

In [ ]:
# --- GIVEN: visualize the similarity matrix ---

encoder_viz = SimCLREncoder().to(device)

with torch.no_grad():
    mini = [pair_dataset[i] for i in range(4)]
    v1 = torch.stack([m[0] for m in mini]).to(device)
    v2 = torch.stack([m[1] for m in mini]).to(device)
    _, z1 = encoder_viz(v1)
    _, z2 = encoder_viz(v2)
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    sim = (z1 @ z2.T)  # cosine similarity after normalization

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(sim.cpu(), cmap='RdBu_r', vmin=-1, vmax=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{sim[i, j].item():.2f}', ha='center', va='center', fontsize=9)
    ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, edgecolor='lime', lw=2))
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels([f'z2[{i}]' for i in range(4)], fontsize=9)
ax.set_yticklabels([f'z1[{i}]' for i in range(4)], fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('cos(z1[i], z2[j])  —  green = positive pair')
plt.tight_layout()
plt.show()

Each **row** is a question: *"which column is my positive?"*

Row 0's positive is column 0, row 1's is column 1, … — always the **diagonal**. So the labels are simply `[0, 1, 2, …, N−1]` → `torch.arange(N)`.

Now notice: `F.cross_entropy(sim / τ, labels)` computes exactly

$$-\log \frac{\exp(\text{sim}[i, i] / \tau)}{\sum_j \exp(\text{sim}[i, j] / \tau)}$$

**That IS InfoNCE.** Cosine similarity → scale by temperature → cross entropy. No masking, no tricks.

### Implementation plan

1. **Normalize** `z1` and `z2` so that dot product = cosine similarity
2. Compute the **similarity matrix** `sim[i][j] = cos(z1[i], z2[j])`
3. The positive for `z1[i]` is `z2[i]` → label = `i`
4. `F.cross_entropy(sim / τ, labels)`

In [ ]:
# ============================================================
# TODO: Implement InfoNCE loss
# ============================================================

def info_nce(z1, z2, temperature=0.5):
    """
    z1, z2: (N, D) embeddings from two views of the same batch.
    z1[i] and z2[i] are a positive pair.
    """
    # Step 1: L2-normalize so that dot product = cosine similarity
    z1 = F.normalize(z1, dim=None)  # TODO
    z2 = F.normalize(z2, dim=None)  # TODO

    # Step 2: cosine similarity matrix  sim[i][j] = cos(z1[i], z2[j])
    sim = None  # TODO

    # Step 3: scale by temperature
    sim = sim / temperature

    # Step 4: positive for row i is column i → labels = [0, 1, …, N-1]
    labels = torch.arange(None, device=z1.device)  # TODO

    # cross_entropy = -log(exp(pos) / sum(exp(all))) = InfoNCE!
    return F.cross_entropy(None, None)  # TODO

In [ ]:
# --- GIVEN: sanity check ---
N = 256
z_test = torch.randn(N, 128, device=device)

random_loss   = info_nce(z_test, torch.randn(N, 128, device=device)).item()
identical_loss = info_nce(z_test, z_test.clone()).item()

print(f'Random pairs   → loss = {random_loss:.2f}  (expect ≈ ln({N}) = {np.log(N):.2f})')
print(f'Identical pairs → loss = {identical_loss:.2f}  (should be much lower than {np.log(N):.2f})')

---
## Contrastive Training

We train for **10 epochs** on 20K unlabeled images. No labels needed!

In [ ]:
# ============================================================
# TODO: Fill in the training loop
# ============================================================

encoder = SimCLREncoder().to(device)
optimizer = optim.Adam(encoder.parameters(), lr=3e-4, weight_decay=1e-4)
losses = []

encoder.train()
for epoch in range(10):
    epoch_loss = 0
    for v1, v2 in tqdm(pair_loader, desc=f'Epoch {epoch+1}', leave=False):
        v1, v2 = v1.to(device), v2.to(device)

        _, z1 = encoder(None)  # TODO
        _, z2 = encoder(None)  # TODO

        loss = info_nce(None, None)  # TODO

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg = epoch_loss / len(pair_loader)
    losses.append(avg)
    print(f'Epoch {epoch+1:2d} | Loss {avg:.3f}')

In [ ]:
# --- GIVEN ---
plt.figure(figsize=(7, 3.5))
plt.plot(range(1, len(losses) + 1), losses, 'b-o', markersize=5)
plt.axhline(np.log(256), color='r', ls='--', alpha=0.4, label=f'random baseline (ln 256 ≈ {np.log(256):.1f})')
plt.xlabel('Epoch')
plt.ylabel('InfoNCE Loss')
plt.legend()
plt.title('Contrastive Training')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Who Learns Better Features?

The big question: is contrastive pretraining actually useful?

We compare two ResNet-18 models, both evaluated on **1,000 labeled images** (100 per class):

| Model | Backbone init | What trains | Epochs |
|-------|--------------|-------------|--------|
| **From scratch** | Random weights | Everything (backbone + classifier) | 10 |
| **Contrastive** | Our pretrained weights | Only the classifier head (backbone **frozen**) | 10 |

If contrastive pretraining learned good features, a simple linear head on frozen features should beat training the whole model from scratch with limited labels.

In [ ]:
# --- GIVEN: evaluation setup ---

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

cifar_train_eval = torchvision.datasets.CIFAR10(root='./data', train=True,  transform=eval_transform)
cifar_test_eval  = torchvision.datasets.CIFAR10(root='./data', train=False, transform=eval_transform)

# 1K labeled images (100 per class) — simulates scarce labels
indices_1k = []
counts = [0] * 10
for i in range(len(cifar_train_eval)):
    _, label = cifar_train_eval[i]
    if counts[label] < 100:
        indices_1k.append(i)
        counts[label] += 1
    if sum(counts) == 1000:
        break

train_labeled = Subset(cifar_train_eval, indices_1k)
train_loader_eval = DataLoader(train_labeled, batch_size=256, shuffle=True, num_workers=0)
test_loader_eval  = DataLoader(cifar_test_eval, batch_size=256, num_workers=0)

print(f'Labeled train: {len(train_labeled):,}  |  Test: {len(cifar_test_eval):,}')


def train_and_evaluate(model, train_ldr, test_ldr, epochs=10, lr=1e-3):
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    model.train()
    for ep in range(epochs):
        for imgs, labels in train_ldr:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = F.cross_entropy(model(imgs), labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_ldr:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
# --- GIVEN: train both models ---

# Model A: ResNet-18 from scratch — train everything
print('Model A: ResNet-18 from scratch (10 epochs, 1K labels)...')
scratch_model = models.resnet18(weights=None)
scratch_model.fc = nn.Linear(512, 10)
scratch_model = scratch_model.to(device)
scratch_acc = train_and_evaluate(scratch_model, train_loader_eval, test_loader_eval)
print(f'  → Test accuracy: {scratch_acc:.1%}\n')

# Model B: contrastive backbone (frozen) + linear head
print('Model B: Contrastive backbone + linear head (10 epochs, 1K labels)...')
contrastive_clf = nn.Sequential(
    encoder.backbone,
    nn.Flatten(),
    nn.Linear(512, 10),
).to(device)

for p in contrastive_clf[0].parameters():
    p.requires_grad = False

contrastive_acc = train_and_evaluate(contrastive_clf, train_loader_eval, test_loader_eval)
print(f'  → Test accuracy: {contrastive_acc:.1%}')

In [ ]:
# --- GIVEN: comparison chart ---

fig, ax = plt.subplots(figsize=(6, 4))
names = ['From Scratch\n(full model)', 'Contrastive\n(frozen backbone\n+ linear head)']
accs = [scratch_acc * 100, contrastive_acc * 100]
colors = ['#d63031', '#0984e3']
bars = ax.bar(names, accs, color=colors, edgecolor='k', width=0.5)
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5,
            f'{b.get_height():.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('Test Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('Supervised From Scratch  vs  Contrastive Pretraining\n(1,000 labeled images, 10 epochs each)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

diff = contrastive_acc - scratch_acc
if diff > 0:
    print(f'Contrastive pretraining wins by {diff*100:.1f}% — and only a linear layer was trained!')
else:
    print(f'From scratch wins by {-diff*100:.1f}% — try more contrastive epochs or more unlabeled data.')

---
## Discussion

1. **Why strong augmentations?** What happens if we only use random crops (no colour jitter, no blur)?

2. **Projection head paradox:** We compute the loss on **z** but keep **h** for downstream. Why throw away **z**?

3. **Temperature τ:** What happens when τ is very large (10.0) vs very small (0.01)?

4. **Batch size:** Each image in the batch acts as a negative for every other. What happens with tiny batches?

5. **Real-world use:** You have 100 labeled and 50,000 unlabeled medical images. How would you use contrastive learning?

---
## Wrap-Up

| Concept | What You Learned |
|---|---|
| Pair Dataset | A custom `Dataset` returning two augmented views of the same image |
| Augmentations | They define the learning signal — strong augmentations force deep features |
| SimCLR Encoder | Backbone (features we keep) + projection head (for the loss only) |
| InfoNCE | Cosine similarity → temperature scaling → cross entropy. No built-in! |
| Pretraining value | Contrastive features + linear head can beat training from scratch |